# Experiment 2 — LLM sense-assignment results

The **model side** of Experiment 2: how the 5 deployable LLMs perform at
assigning SLS senses to Neo-Latin corpus lines, scored against the
human-annotated gold built in
[`exp2_iaa.ipynb`](exp2_iaa.ipynb). Structured to mirror the Exp 1 result
notebooks (`exp1_eval_analysis` + `exp1_phase_b_analysis`) so the two
experiments are directly comparable.

**Sections**
- §1 scope · §2 per-provider×config headline (the paper table) · §3 MFS
  baselines · §4 prompt-template effects · §5 axis effects (def_language
  headline; context/metadata EXTENDED) · §6 per-lemma & cross-lemma · §7
  per-sense P/R/F1 + confusion · **§8 LLM vs human strata** (novelty/`0`,
  confidence, genuine-ambiguity, adjudicated) · §9 sentence-length & context ·
  §10 error analysis · §11 exp1↔exp2 comparison · §12 paper outputs.

**Two correctness facts that shape every metric here:**
1. **The corpus is most-frequent-sense-dominated** (macro MFS ≈ 0.82; see
   `exp2_iaa.ipynb` §2). Accuracy must always be read against this high floor —
   unlike Exp 1's 48.7% MFS. The lexicographic value is concentrated in the hard
   and novel cases, not the headline accuracy.
2. **Candidate-sense numbering is shuffled per record** (seed = record_id), so a
   predicted *number* means different senses across records. Accuracy is
   unaffected (predicted==gold within a record), but κ, confusion matrices and
   per-sense P/R/F1 use the **reconstructed sense ids** (`predicted_sid`, via
   `exp2_attach_sense_ids`). Sid-level pooled κ is lemma-blocked just like the
   annotator κ, so the two are directly comparable.

**Runs:** headline = CORE (`exp2_core`, complete). EXTENDED (`exp2_extended`)
is not yet run at full scale — its sections degrade gracefully. See
[`exp2_runs.md`](exp2_runs.md).

## §0. Setup, load & derived columns

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd() if (Path.cwd() / "_eval_loader.py").exists() else Path.cwd() / "analysis"
sys.path.insert(0, str(HERE))
import _eval_loader as E
from _eval_loader import (eval_id, EVALS, REPO_ROOT, EXP2_EVAL_SET_CSV, EXP2_GOLD_CSV,
                          EXP2_LEMMAS, EXP2_TIER_A, EXP2_TIER_B,
                          bootstrap_ci_binary, mcnemar, holm_bonferroni,
                          kappa_labels, exp2_mfs_baseline, exp2_lemma_features,
                          exp2_attach_sense_ids)

RNG_SEED, BOOTSTRAP_N = 20260516, 10_000
TIER = {l: ("A" if l in EXP2_TIER_A else "B") for l in EXP2_LEMMAS}
PROVIDER_ORDER = ["gemma-4-31b-it", "qwen3-235b-a22b-2507", "llama-4-maverick",
                  "deepseek-chat-v3-0324", "llama-3.3-70b-instruct"]
TEMPLATES = ["zeroshot", "fewshot", "justified", "decomposed"]

# Load CORE (always). EXTENDED only if registered with a non-null id.
def _has(name):
    try:
        eval_id(name); return True
    except (KeyError, ValueError):
        return False

core_meta, df_all = E.load_exp2_eval(eval_id("exp2_core"))
df_all = exp2_attach_sense_ids(df_all)       # adds predicted_sid (sense-level)
HAS_EXTENDED = _has("exp2_extended")
if HAS_EXTENDED:
    _, dfe = E.load_exp2_eval(eval_id("exp2_extended"))
    dfe = exp2_attach_sense_ids(dfe)
    df_all = pd.concat([df_all, dfe], ignore_index=True)   # CORE ∪ EXTENDED = full grid

# Full-grid config (template+def, plus context/metadata when they vary) — used
# only by the EXTENDED-axis cell in §5.
EXT_AXES = [a for a in ("context_length", "show_metadata") if df_all[a].nunique() > 1]
def _full_config(r):
    return "|".join([r["template"], f"def={r['def_language']}"] + [f"{a}={r[a]}" for a in EXT_AXES])
df_all["config"] = df_all.apply(_full_config, axis=1)
df_all["lemma_uc"] = df_all["lemma"].str.upper()

# Source metadata (genre/author/period) on siglum — error covariates for §10.
# Only 4 sources (3 treatises + 1 polemic, all ~1660s–1670s, 3 authors): COARSE
# and confounded with lemma (some lemmas are single-source). Period is ~uniform.
_meta = pd.read_csv(Path(os.environ["DIALEXPOL_DATA_DIR"]) / "neolatin_concordances" / "sources.csv",
                    dtype=str, keep_default_na=False)[["siglum", "author", "time", "genre", "domain"]]
df_all = df_all.merge(_meta, on="siglum", how="left")

# HEADLINE FRAME `df` = the exp1-analogous CORE condition (kwic_only, meta off).
# Every headline section below uses `df`; ONLY the §5 EXTENDED cell uses df_all.
# Scenarios are disjoint, so the CORE condition's cells come solely from the CORE
# run; when EXTENDED isn't loaded, df == df_all (identical output to a CORE-only run).
CORE_MASK = (df_all["context_length"] == "kwic_only") & (~df_all["show_metadata"].astype(bool))
df = df_all[CORE_MASK].copy()
df["config"] = df["template"] + "|def=" + df["def_language"]   # clean 8-config headline id

# EXTENDED completeness guard — the extended run fills progressively, so its
# numbers are PRELIMINARY until it reaches the full grid (6 non-core axis cells ×
# 4 templates × n_providers × 837 records). §5 flags partial results loudly.
EXPECTED_EXT_ROWS = df_all["provider_label"].nunique() * df_all["template"].nunique() * 6 * df["key"].nunique()
ext_rows = int((~CORE_MASK).sum())
ext_records = int(df_all.loc[~CORE_MASK, "key"].nunique()) if HAS_EXTENDED else 0
EXT_COMPLETE = HAS_EXTENDED and ext_rows >= EXPECTED_EXT_ROWS

# MFS baseline (per-lemma + macro + micro). The micro rate weights by lemma size.
mfs_per, mfs_macro = exp2_mfs_baseline()
_ev = pd.read_csv(EXP2_EVAL_SET_CSV, dtype=str, keep_default_na=False)
_mfs_sid = _ev.groupby("lemma")["gold_sense_id"].agg(lambda s: s.value_counts().idxmax())
mfs_micro = _ev.apply(lambda r: r["gold_sense_id"] == _mfs_sid[r["lemma"]], axis=1).mean()
LEMMA_FEATS = exp2_lemma_features()

print(f"CORE: {core_meta['id']}")
print(f"  headline frame `df` (CORE: kwic_only, meta off): {len(df):,} rows, {df['config'].nunique()} configs, {df['key'].nunique()} records")
if HAS_EXTENDED:
    state = "COMPLETE" if EXT_COMPLETE else (f"IN PROGRESS / PARTIAL — {ext_records}/{df['key'].nunique()} records, "
                                            f"{ext_rows:,}/{EXPECTED_EXT_ROWS:,} rows")
    print(f"  full grid `df_all`: {len(df_all):,} rows, {df_all['config'].nunique()} configs — EXTENDED {state}")
    if not EXT_COMPLETE:
        print("  >>> EXTENDED still filling: §5 extended-axis numbers are PRELIMINARY and WILL change.")
        print("  >>> Re-run after it finishes. The CORE headline (§2–§4, §6–§11) is unaffected.")
else:
    print(f"  full grid `df_all`: {len(df_all):,} rows — EXTENDED not registered (CORE only)")
print(f"MFS baseline: macro={mfs_macro:.3f}  micro={mfs_micro:.3f}")

## §1. What this eval did (scope & outcomes)

In [ ]:
print("SCOPE — headline = CORE (kwic_only, meta off)\n")
print(f"  providers ({df.provider_label.nunique()}): {sorted(df.provider_label.unique())}")
print(f"  templates ({df.template.nunique()}): {sorted(df.template.unique())}")
print(f"  def_language: {sorted(df.def_language.unique())}  (context_length / show_metadata pinned in CORE)")
print(f"  records: {df.key.nunique()}  ×  {df.config.nunique()} configs  ×  {df.provider_label.nunique()} providers = {len(df):,} cells")
print(f"\n  outcomes: {df.outcome.value_counts().to_dict()}")
print(f"  overall accuracy (pooled): {df.correct.mean():.4f}")
api_err = (df.outcome.eq('wrong') & df.predicted.isna()).sum()
print(f"  format failures: {(df.outcome=='format_failure').sum()}  |  empty/API-error rows (counted wrong): {api_err}")
print(f"  total spend (CORE): ${df.cost.sum():.3f}  over {len(df):,} calls  ({1e3*df.cost.sum()/len(df):.3f} m$/call)")
if HAS_EXTENDED:
    ex = df_all[~CORE_MASK]
    print(f"\n  EXTENDED loaded: +{len(ex):,} cells over {ex.config.nunique()} extra configs "
          f"(context_length={sorted(df_all.context_length.unique())}, show_metadata={sorted(df_all.show_metadata.unique())}).")
    print(f"  Extended axes are analysed in §5b only; the headline below stays CORE.")
print(f"\n  MFS floor: macro {mfs_macro:.3f} / micro {mfs_micro:.3f}  — accuracy is read against this.")

## §2. Per-provider × config headline (the paper table)

Per (provider, config): accuracy ± bootstrap 95% CI (10K resamples), Cohen's κ
vs gold (**sense-id level**, comparable to the annotator κ), Δ vs micro-MFS,
format-failure rate, spend and $/correct. `clears_mfs` flags configs whose CI
lower bound exceeds the micro-MFS floor — a deliberately stringent bar given
the MFS-dominated corpus.

In [ ]:
rng = np.random.default_rng(RNG_SEED)
rows = []
for (prov, cfg), g in df.groupby(["provider_label", "config"]):
    n, k = len(g), int(g.correct.sum())
    lo, hi = bootstrap_ci_binary(k, n, n_boot=BOOTSTRAP_N, rng=rng)
    rows.append({
        "provider": prov, "config": cfg, "n": n,
        "acc%": round(100 * k / n, 1), "ci_lo%": round(100 * lo, 1), "ci_hi%": round(100 * hi, 1),
        "kappa": round(kappa_labels(g.predicted_sid, g.gold_sense_id), 3),
        "dMFS_pp": round(100 * (k / n - mfs_micro), 1),
        "ff%": round(100 * (g.outcome == "format_failure").mean(), 2),
        "spent$": round(g.cost.sum(), 3),
        "$/corr": round(g.cost.sum() / max(k, 1), 5),
        "clears_mfs": lo > mfs_micro,
    })
hd = pd.DataFrame(rows).sort_values(["provider", "acc%"], ascending=[True, False])
hd["provider"] = pd.Categorical(hd["provider"], PROVIDER_ORDER, ordered=True)
hd = hd.sort_values(["provider", "acc%"], ascending=[True, False]).reset_index(drop=True)
print("Per-(provider, config) headline — CORE\n")
with pd.option_context("display.width", 200, "display.max_rows", 60):
    print(hd.to_string(index=False))
print(f"\nconfigs clearing micro-MFS ({mfs_micro:.3f}) at CI lower bound: "
      f"{int(hd.clears_mfs.sum())}/{len(hd)}")
HEADLINE = hd

In [ ]:
# Per-provider best config + forest plot (accuracy ± CI).
best = (hd.sort_values("acc%", ascending=False).groupby("provider", observed=True).first()
          .reindex(PROVIDER_ORDER))
fig, ax = plt.subplots(figsize=(9, 4.5))
y = np.arange(len(best))
ax.errorbar(best["acc%"], y, xerr=[best["acc%"]-best["ci_lo%"], best["ci_hi%"]-best["acc%"]],
            fmt="o", color="#4c72b0", ecolor="#999", capsize=3)
ax.axvline(100*mfs_micro, ls="--", c="#c44e52", lw=1.2, label=f"micro-MFS {100*mfs_micro:.1f}%")
ax.axvline(100*mfs_macro, ls=":", c="#dd8452", lw=1.2, label=f"macro-MFS {100*mfs_macro:.1f}%")
ax.set_yticks(y); ax.set_yticklabels([f"{p}\n({c})" for p, c in zip(best.index, best["config"])], fontsize=8)
ax.set_xlabel("accuracy % (best config per provider, 95% bootstrap CI)")
ax.set_title("Exp 2 CORE — best config per provider vs MFS floor"); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()
print(best[["config", "acc%", "ci_lo%", "ci_hi%", "kappa", "dMFS_pp"]].to_string())

In [ ]:
# "Tied for best": configs whose 95% bootstrap CI overlaps the single best cell's
# CI — statistically indistinguishable from the top at this N (exp1 §7).
best_row = hd.sort_values("acc%", ascending=False).iloc[0]
tied = hd[hd["ci_hi%"] >= best_row["ci_lo%"]].sort_values("acc%", ascending=False)
print(f"Best cell: {best_row['provider']} / {best_row['config']} = {best_row['acc%']}% "
      f"[{best_row['ci_lo%']}, {best_row['ci_hi%']}]\n")
print(f"Configs whose CI overlaps the best (not significantly worse): {len(tied)}/{len(hd)}\n")
print(tied[["provider", "config", "acc%", "ci_lo%", "ci_hi%", "kappa"]].to_string(index=False))
print(f"\nProviders represented in the tied set: {sorted(tied.provider.unique().tolist())}")
print("Softer ranking than the point estimates; exp2 has no Phase-A→B promotion gate, so this")
print("is descriptive (which setups a reader should treat as interchangeable for deployment).")

## §3. Baselines (most-frequent-sense)

The MFS floor per lemma and how often the best model beats it. With a macro MFS
of ~0.82, MFS is a strong baseline; the interesting signal is *where* models
beat it and *by how much* on the polysemous, low-MFS lemmas.

In [ ]:
best_prov = hd.iloc[hd["acc%"].idxmax()]["provider"]
best_cfg = hd.iloc[hd["acc%"].idxmax()]["config"]
bsub = df[(df.provider_label == best_prov) & (df.config == best_cfg)]
per_lemma_acc = bsub.groupby("lemma").correct.mean()
base = pd.DataFrame({"MFS": pd.Series(mfs_per), "best_LLM_acc": per_lemma_acc})
base["n"] = df[df.eval_id == core_meta["id"]].groupby("lemma").key.nunique() if False else _ev.groupby("lemma").size()
base["L1_senses"] = LEMMA_FEATS["n_candidate_senses"]
base["beats_MFS_pp"] = (100 * (base.best_LLM_acc - base.MFS)).round(1)
base = base.sort_values("MFS")
print(f"Best config = {best_prov} / {best_cfg}\n")
print(base.round(3).to_string())
print(f"\nlemmas where best LLM beats MFS: {(base.beats_MFS_pp > 0).sum()}/{len(base)}; "
      f"macro best-LLM acc {base.best_LLM_acc.mean():.3f} vs macro MFS {mfs_macro:.3f}; "
      f"micro best-LLM acc {bsub.correct.mean():.3f} vs micro MFS {mfs_micro:.3f}")
print("Read: gains concentrate on low-MFS polysemous lemmas (officium, libertas, praefectus);")
print("on near-deterministic lemmas (comitia, lex, imperator) MFS is already a ceiling.")

## §4. Prompt-template effects

Marginal accuracy by template (pooled over providers and def_language) and
pairwise McNemar's tests on paired records, Holm-corrected across the 6 template
pairs.

In [ ]:
def paired_mcnemar(frame, col, v1, v2, within):
    a = frame[frame[col] == v1].set_index(within)["correct"]
    b = frame[frame[col] == v2].set_index(within)["correct"]
    j = a.to_frame("a").join(b.to_frame("b"), how="inner").dropna()
    B = int((j.a & ~j.b).sum()); C = int((~j.a & j.b).sum())
    chi2, p = mcnemar(B, C)
    return len(j), B, C, p

tmpl = df.groupby("template").correct.agg(["mean", "size"]).reindex(TEMPLATES)
tmpl["acc%"] = (100 * tmpl["mean"]).round(2)
print("Marginal accuracy by template (pooled)\n")
print(tmpl[["acc%", "size"]].to_string())

pairs, pvals, recs = [], [], []
for i in range(len(TEMPLATES)):
    for jx in range(i + 1, len(TEMPLATES)):
        t1, t2 = TEMPLATES[i], TEMPLATES[jx]
        npair, B, C, p = paired_mcnemar(df, "template", t1, t2,
                                        ["provider_label", "def_language", "context_length", "show_metadata", "key"])
        pairs.append(f"{t1} vs {t2}"); pvals.append(p)
        recs.append({"pair": f"{t1} vs {t2}", "n_pairs": npair, "b": B, "c": C,
                     "acc1%": round(100*tmpl.loc[t1,'mean'],1), "acc2%": round(100*tmpl.loc[t2,'mean'],1), "p": p})
padj = holm_bonferroni(pvals)
mc = pd.DataFrame(recs); mc["p_holm"] = [round(x, 4) for x in padj]; mc["p"] = mc["p"].round(4)
print("\nPairwise McNemar (Holm-corrected across 6 pairs)\n")
print(mc.to_string(index=False))

In [ ]:
# Template marginal accuracy with bootstrap CIs (chart for §4).
rng = np.random.default_rng(RNG_SEED)
accs, los, his = [], [], []
for t in TEMPLATES:
    g = df[df.template == t]; k = int(g.correct.sum()); nn = len(g)
    a = k / nn; lo, hi = bootstrap_ci_binary(k, nn, n_boot=BOOTSTRAP_N, rng=rng)
    accs.append(100 * a); los.append(100 * (a - lo)); his.append(100 * (hi - a))
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(TEMPLATES, accs, yerr=[los, his], capsize=5, color="#4c72b0")
ax.axhline(100 * mfs_micro, ls="--", c="#c44e52", lw=1, label=f"micro-MFS {100*mfs_micro:.1f}%")
for i, a in enumerate(accs):
    ax.text(i, a + 1.2, f"{a:.1f}", ha="center", fontsize=8)
ax.set_ylabel("accuracy %"); ax.set_ylim(60, 90)
ax.set_title("Prompt-template marginal accuracy (pooled, ±95% CI)")
ax.legend(fontsize=8); fig.tight_layout(); plt.show()

## §5. Axis effects

`def_language` is the **headline axis** (replicates the Exp 1 la↔pl comparison),
analysed below on the CORE slice. The **EXTENDED** axes `context_length` (+prev
sentence) and `show_metadata` get their own results in **§5b** (preliminary /
future-work), shown once the extended run is registered.

In [ ]:
# def_language: marginal + paired McNemar (pooled and per provider).
dl = df.groupby("def_language").correct.mean().mul(100).round(2)
n, B, C, p = paired_mcnemar(df, "def_language", "la", "pl",
                            ["provider_label", "template", "context_length", "show_metadata", "key"])
print("def_language marginal accuracy (pooled): " + ", ".join(f"{k}={v}%" for k, v in dl.items()))
print(f"paired McNemar la vs pl: n={n}, b(la✓pl✗)={B}, c(la✗pl✓)={C}, p={p:.4g}  "
      f"(Δ = {dl.get('la',0)-dl.get('pl',0):+.2f} pp la−pl)")
print("\nper-provider la vs pl:")
for prov, g in df.groupby("provider_label"):
    la = g[g.def_language=="la"].correct.mean(); pl = g[g.def_language=="pl"].correct.mean()
    _, b, c, pp = paired_mcnemar(g, "def_language", "la", "pl",
                                 ["template", "context_length", "show_metadata", "key"])
    print(f"  {prov:24s} la={100*la:5.1f}%  pl={100*pl:5.1f}%  Δ={100*(la-pl):+5.1f}pp  p={pp:.3g}")

In [ ]:
# def_language effect charts: marginal (±CI) + per-provider grouped bars.
rng = np.random.default_rng(RNG_SEED)
vals = ["la", "pl"]; colors = {"la": "#dd8452", "pl": "#4c72b0"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
accs, los, his = [], [], []
for v in vals:
    g = df[df.def_language == v]; k = int(g.correct.sum()); nn = len(g)
    a = k / nn; lo, hi = bootstrap_ci_binary(k, nn, n_boot=BOOTSTRAP_N, rng=rng)
    accs.append(100 * a); los.append(100 * (a - lo)); his.append(100 * (hi - a))
axes[0].bar(vals, accs, yerr=[los, his], capsize=6, color=[colors[v] for v in vals])
axes[0].axhline(100 * mfs_micro, ls="--", c="#c44e52", lw=1, label="micro-MFS")
for i, a in enumerate(accs):
    axes[0].text(i, a + 1, f"{a:.1f}", ha="center", fontsize=9)
axes[0].set_title("(a) def_language marginal (±95% CI)"); axes[0].set_ylabel("accuracy %")
axes[0].set_ylim(60, 90); axes[0].legend(fontsize=8)
x = np.arange(len(PROVIDER_ORDER)); w = 0.38
for i, v in enumerate(vals):
    pa = [100 * df[(df.provider_label == p) & (df.def_language == v)].correct.mean() for p in PROVIDER_ORDER]
    axes[1].bar(x + (i - 0.5) * w, pa, w, label=v, color=colors[v])
axes[1].set_xticks(x); axes[1].set_xticklabels(PROVIDER_ORDER, rotation=40, ha="right", fontsize=7)
axes[1].set_title("(b) def_language by provider"); axes[1].set_ylabel("accuracy %"); axes[1].legend(fontsize=8)
fig.tight_layout(); plt.show()
print("Polish definitions beat Latin for every provider — the headline axis effect.")

### §5b. EXTENDED scenario results — context length & source metadata

The two **EXTENDED** axes, on the full 32-cell grid (CORE ∪ EXTENDED). Reported
as **preliminary / future-work**: the paper headline stays CORE (§2). The +prev
(context) effect is shown on the records that *have* a previous sentence, and on
the cleaner fully-untruncated subset; `show_metadata` (the one-sentence
provenance line) is the diachronic-cue probe. These render only when
`exp2_extended` is registered (`HAS_EXTENDED`); §0 prints whether the run is
complete.

In [ ]:
# EXTENDED axes — context_length (+prev) and show_metadata, on the FULL grid
# (df_all). PRELIMINARY / future-work (exp2_runs.md), not headline. Activates
# automatically once `exp2_extended` is registered with a real eval id.
if HAS_EXTENDED and df_all["context_length"].nunique() > 1:
    if not EXT_COMPLETE:
        bar = "=" * 74
        print(bar)
        print("INCOMPLETE EXTENDED RUN — the numbers below are PRELIMINARY (partial data).")
        print(f"  covered {ext_records}/{df['key'].nunique()} records, {ext_rows:,}/{EXPECTED_EXT_ROWS:,} rows"
              f"  ({sorted(df_all.loc[~CORE_MASK,'lemma'].unique())}).")
        print("  Do not cite; re-run after the extended eval finishes for final figures.")
        print(bar + "\n")
    print("EXTENDED axes (PRELIMINARY — paired McNemar on the full grid)\n")
    # context_length (+prev): a previous sentence can only help records that HAVE
    # one — restrict the test to prev-present records, then to the cleanest
    # fully-untruncated subset (analysis plan §7 context-availability stratum).
    has_prev = df_all["prev_sent"].fillna("").str.strip() != ""
    sub = df_all[has_prev]
    marg = sub.groupby("context_length").correct.mean().mul(100).round(2)
    n, B, C, p = paired_mcnemar(sub, "context_length", "kwic_only", "kwic_plus_prev",
                                ["provider_label", "template", "def_language", "show_metadata", "key"])
    d = marg.get("kwic_plus_prev", float("nan")) - marg.get("kwic_only", float("nan"))
    print(f"context_length [records with a prev sentence, n={sub.key.nunique()}]: {dict(marg)}")
    print(f"  McNemar kwic_only vs +prev: b={B} c={C} p={p:.4g}  (Δ={d:+.2f}pp, +prev − kwic_only)")
    unt = df_all[df_all.context_truncated == "none"]
    mu = unt.groupby("context_length").correct.mean().mul(100).round(2)
    print(f"  [fully-untruncated subset, n={unt.key.nunique()}]: {dict(mu)}")
    # show_metadata (off vs on)
    if df_all["show_metadata"].nunique() > 1:
        mm = df_all.groupby("show_metadata").correct.mean().mul(100).round(2)
        n, B, C, p = paired_mcnemar(df_all, "show_metadata", False, True,
                                    ["provider_label", "template", "def_language", "context_length", "key"])
        print(f"\nshow_metadata: {dict(mm)}  | McNemar off vs on: b={B} c={C} p={p:.4g}")
    print("\nBoth axes are statistically SIGNIFICANT but MODEST in magnitude (~1pp pooled, up to ~2pp")
    print("on gemma); the tiny p-values are a big-N effect (≈50–67K paired records), not a large effect.")
    print("Note +metadata (~+1pp; +2.1 on gemma) EXCEEDS exp1 Phase A's +0.43pp 'operationally null' —")
    print("source metadata helps MORE on corpus than on dictionary quotes. Still preliminary/future-work;")
    print("the +prev effect rests on the untruncated subset and the headline stays CORE.")
else:
    print("EXTENDED axes (context_length=+prev, show_metadata=on) not yet loaded.")
    print("This cell activates automatically once `exp2_extended` is registered with a real")
    print("eval id (see exp2_runs.md). Until then, headline = CORE (kwic_only, meta off).")

In [ ]:
# EXTENDED effect charts: context_length on untruncated records, show_metadata
# pooled, and the 2×2 context×metadata accuracy view.
if HAS_EXTENDED and df_all["context_length"].nunique() > 1:
    rng = np.random.default_rng(RNG_SEED)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    unt = df_all[df_all.context_truncated == "none"]
    ax = axes[0]
    for i, v in enumerate(["kwic_only", "kwic_plus_prev"]):
        g = unt[unt.context_length == v]; k = int(g.correct.sum()); nn = len(g)
        a = k / nn; lo, hi = bootstrap_ci_binary(k, nn, n_boot=BOOTSTRAP_N, rng=rng)
        ax.bar(i, 100 * a, yerr=[[100 * (a - lo)], [100 * (hi - a)]], capsize=6,
               color=["#4c72b0", "#55a868"][i])
        ax.text(i, 100 * a + 0.5, f"{100*a:.1f}", ha="center", fontsize=9)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["kwic_only", "+prev"]); ax.set_ylim(70, 86)
    ax.set_title(f"(a) context_length\n(untruncated, n={unt.key.nunique()} records)"); ax.set_ylabel("accuracy %")
    ax = axes[1]
    for i, v in enumerate([False, True]):
        g = df_all[df_all.show_metadata.astype(bool) == v]; k = int(g.correct.sum()); nn = len(g)
        a = k / nn; lo, hi = bootstrap_ci_binary(k, nn, n_boot=BOOTSTRAP_N, rng=rng)
        ax.bar(i, 100 * a, yerr=[[100 * (a - lo)], [100 * (hi - a)]], capsize=6,
               color=["#bbbbbb", "#dd8452"][i])
        ax.text(i, 100 * a + 0.5, f"{100*a:.1f}", ha="center", fontsize=9)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["meta off", "meta on"]); ax.set_ylim(70, 86)
    ax.set_title("(b) show_metadata (pooled)"); ax.set_ylabel("accuracy %")
    ax = axes[2]
    piv = df_all.pivot_table(index="context_length", columns="show_metadata", values="correct", aggfunc="mean") * 100
    im = ax.imshow(piv.values, cmap="YlGn", aspect="auto")
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels([f"meta={c}" for c in piv.columns], fontsize=8)
    ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index, fontsize=8)
    for ii in range(piv.shape[0]):
        for jj in range(piv.shape[1]):
            ax.text(jj, ii, f"{piv.values[ii,jj]:.1f}", ha="center", va="center", fontsize=10)
    ax.set_title("(c) context × metadata accuracy %"); fig.colorbar(im, ax=ax)
    fig.tight_layout(); plt.show()
else:
    print("EXTENDED not loaded — charts skipped (register exp2_extended; see exp2_runs.md).")

In [ ]:
# Per-provider EXTENDED effects: Δ accuracy from +prev (untruncated) and +metadata.
if HAS_EXTENDED and df_all["context_length"].nunique() > 1:
    unt = df_all[df_all.context_truncated == "none"]
    rows = []
    for p in PROVIDER_ORDER:
        gp = unt[unt.provider_label == p]
        ko = gp[gp.context_length == "kwic_only"].correct.mean()
        kp = gp[gp.context_length == "kwic_plus_prev"].correct.mean()
        gm = df_all[df_all.provider_label == p]
        mo = gm[~gm.show_metadata.astype(bool)].correct.mean()
        mn = gm[gm.show_metadata.astype(bool)].correct.mean()
        rows.append({"provider": p, "+prev Δpp": round(100 * (kp - ko), 1),
                     "+meta Δpp": round(100 * (mn - mo), 1)})
    ed = pd.DataFrame(rows)
    print("Per-provider EXTENDED effects (Δ accuracy, pp)\n")
    print(ed.to_string(index=False))
    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(ed)); w = 0.38
    ax.bar(x - w / 2, ed["+prev Δpp"], w, label="+prev (untruncated)", color="#55a868")
    ax.bar(x + w / 2, ed["+meta Δpp"], w, label="+metadata", color="#dd8452")
    ax.axhline(0, c="k", lw=0.8); ax.set_xticks(x)
    ax.set_xticklabels(ed.provider, rotation=40, ha="right", fontsize=7)
    ax.set_ylabel("Δ accuracy (pp)"); ax.set_title("Per-provider extended-axis effects")
    ax.legend(fontsize=8); fig.tight_layout(); plt.show()
else:
    print("EXTENDED not loaded.")

In [ ]:
# Full-grid top setups + whether the extended axes raise the achievable ceiling.
if HAS_EXTENDED and df_all["context_length"].nunique() > 1:
    cellacc = (df_all.groupby(["provider_label", "template", "def_language",
                               "context_length", "show_metadata"])
               .correct.mean().mul(100).round(1).rename("acc%").reset_index())
    top = cellacc.sort_values("acc%", ascending=False).head(12)
    print("Top 12 full-grid setups (provider × template × def × context × metadata)\n")
    print(top.to_string(index=False))
    best_core = float(HEADLINE["acc%"].max())
    best_full = float(cellacc["acc%"].max())
    print(f"\nBest CORE config {best_core:.1f}%  |  best full-grid config {best_full:.1f}%  (Δ {best_full-best_core:+.1f}pp).")
    print("Pooled marginal effects are small (~+0.8pp +prev, ~+1.0pp +meta; see §5b McNemar), but")
    print("STACKING both favourable axes on the strongest setup (gemma/justified/pl) reaches the")
    print("full-grid best — an optimistic ceiling (max over 4× more cells, so winner's-curse inflated).")
    print("Reported as preliminary future-work; the paper headline stays the CORE best (§2).")
else:
    print("EXTENDED not loaded.")

## §6. Per-lemma & cross-lemma

Provider × lemma accuracy heatmap, then cross-lemma relationships: does LLM
accuracy track sense count and the *human* difficulty (annotator agreement)?
Descriptive (14 points; sense count is partly an editorial artifact).

In [ ]:
# Provider × lemma accuracy heatmap (pooled over configs).
piv = (df.groupby(["provider_label", "lemma"]).correct.mean().mul(100)
         .unstack().reindex(PROVIDER_ORDER))
order_l = base.index.tolist()  # by MFS ascending
piv = piv[order_l]
fig, ax = plt.subplots(figsize=(13, 3.6))
im = ax.imshow(piv.values, aspect="auto", cmap="RdYlGn", vmin=30, vmax=100)
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index, fontsize=8)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.values[i, j]
        if v == v: ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, label="accuracy %"); ax.set_title("Provider × lemma accuracy (lemmas sorted by MFS)")
fig.tight_layout(); plt.show()

In [ ]:
# Cross-lemma: LLM accuracy (pooled over all cells) vs sense count and vs annotator agreement.
gold = pd.read_csv(EXP2_GOLD_CSV, dtype=str, keep_default_na=False)
gold = gold[gold.analyzable == "True"].copy()
def _both_sense_agree(s):
    p = s.split("|")
    return (len(p) == 2 and p[0] != "" and p[1] != "" and p[0] == p[1])
def _both_assigned(s):
    p = s.split("|")
    return (len(p) == 2 and p[0] != "" and p[1] != "")
gold["both_assigned"] = gold.ann_senses.map(_both_assigned)
gold["agree"] = gold.ann_senses.map(_both_sense_agree)
ann_agree = gold[gold.both_assigned].groupby("lemma").agree.mean()

xl = pd.DataFrame({
    "llm_acc": df.groupby("lemma").correct.mean(),
    "n_senses": LEMMA_FEATS["n_candidate_senses"],
    "mfs": pd.Series(mfs_per),
    "ann_agree": ann_agree,
})
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, xcol, xlab in [(axes[0], "n_senses", "L1 candidate senses"),
                       (axes[1], "ann_agree", "annotator sense agreement")]:
    ax.scatter(xl[xcol], 100 * xl["llm_acc"], s=40, c="#4c72b0")
    for l, r in xl.iterrows():
        ax.annotate(l, (r[xcol], 100*r["llm_acc"]), fontsize=7, xytext=(3,3), textcoords="offset points")
    ax.set_xlabel(xlab); ax.set_ylabel("LLM accuracy % (pooled)")
    rho = xl[[xcol, "llm_acc"]].corr(method="spearman").iloc[0, 1]
    ax.set_title(f"LLM accuracy vs {xlab}  (Spearman ρ={rho:.2f})")
fig.tight_layout(); plt.show()
print("More senses → harder (negative ρ). Higher annotator agreement → higher LLM accuracy:")
print("lemmas that are hard for humans tend to be hard for models (shared difficulty signal).")

In [ ]:
# Provider × provider error agreement: do models fail on the SAME records?
# Per-record per-provider accuracy (mean over the 8 CORE configs), then pairwise
# Pearson r across records — high r = shared hard records (systematic difficulty).
M = df.groupby(["key", "provider_label"]).correct.mean().unstack()[PROVIDER_ORDER]
corr = M.corr()
off = corr.values[np.triu_indices(len(corr), 1)]
fig, ax = plt.subplots(figsize=(6.4, 5.2))
im = ax.imshow(corr.values, cmap="viridis", vmin=float(off.min()), vmax=1)
ax.set_xticks(range(len(PROVIDER_ORDER))); ax.set_xticklabels(PROVIDER_ORDER, rotation=40, ha="right", fontsize=7)
ax.set_yticks(range(len(PROVIDER_ORDER))); ax.set_yticklabels(PROVIDER_ORDER, fontsize=7)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.values[i,j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if corr.values[i, j] < 0.75 else "black")
ax.set_title("Provider×provider error agreement\n(Pearson r of per-record accuracy)")
fig.colorbar(im, ax=ax); fig.tight_layout(); plt.show()
n_right = (M > 0.5).sum(axis=1)   # per record, how many of the 5 providers are (mostly) right
print(f"records all 5 providers get (mostly) right: {(n_right==5).sum()} | all 5 wrong: {(n_right==0).sum()} of {n_right.size}")
print(f"mean pairwise r = {off.mean():.2f} — errors are "
      f"{'largely shared (systematic: the same records are hard for everyone)' if off.mean()>0.4 else 'fairly idiosyncratic across models'}.")

## §7. Per-sense P/R/F1 + confusion (sense-id level)

Computed on the **reconstructed** `predicted_sid` (numbers are per-record
shuffled). Pooled over all provider×config cells — a descriptive view of which
senses are systematically confused. Shown for the polysemous lemmas where this
is informative; with the editorial-artifact caveat (sense granularity is partly
a dictionary-structure choice).

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def per_sense_prf1(lemma):
    g = df[(df.lemma == lemma) & df.predicted_sid.notna()]
    labels = sorted(set(g.gold_sense_id) | set(g.predicted_sid))
    P, R, F, S = precision_recall_fscore_support(g.gold_sense_id, g.predicted_sid,
                                                 labels=labels, zero_division=0)
    out = pd.DataFrame({"sid": labels, "P": P.round(2), "R": R.round(2), "F1": F.round(2),
                        "support_cells": S})
    attested = out[out.support_cells > 0].sort_values("F1")   # senses that occur as gold
    n_unused = (out.support_cells == 0).sum()                 # inventory senses never gold here
    return attested, n_unused

for lemma in ["officium", "libertas", "civitas"]:
    tbl, n_unused = per_sense_prf1(lemma)
    print(f"\n=== {lemma} — per-sense P/R/F1 (attested senses, worst-F1 first) ===")
    print(tbl.to_string(index=False))
    print(f"   (+{n_unused} inventory senses never attested as gold in the sample — omitted)")
print("\n(support_cells = provider×config×def cells, not records; civitas sid '0' = novelty.)")

In [ ]:
# Confusion heatmaps (gold sid → predicted sid, row-normalised, pooled) for the
# lemmas where confusion is most informative: ≥3 attested gold senses, ordered by
# lowest best-config accuracy (the hardest). One figure per lemma (sizes vary).
attested = df.groupby("lemma").gold_sense_id.nunique()
hard_lemmas = (base[base.index.isin(attested[attested >= 3].index)]
               .sort_values("best_LLM_acc").index.tolist()[:6])
print("Confusion shown for (hardest lemmas with ≥3 attested senses):", hard_lemmas)
for lemma in hard_lemmas:
    g = df[(df.lemma == lemma) & df.predicted_sid.notna()]
    cm = pd.crosstab(g.gold_sense_id, g.predicted_sid, normalize="index")
    fig, ax = plt.subplots(figsize=(min(1.0 * len(cm.columns) + 2.5, 12),
                                    min(0.55 * len(cm.index) + 1.6, 8)))
    im = ax.imshow(cm.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(cm.columns))); ax.set_xticklabels(cm.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(len(cm.index))); ax.set_yticklabels(cm.index, fontsize=7)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            v = cm.values[i, j]
            if v >= 0.10:
                ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=6,
                        color="white" if v > 0.5 else "black")
    ax.set_xlabel("predicted sid"); ax.set_ylabel("gold sid")
    ax.set_title(f"{lemma}: confusion (best LLM acc {100*base.loc[lemma,'best_LLM_acc']:.0f}%, diagonal = correct)")
    fig.colorbar(im, ax=ax, label="P(pred | gold)"); fig.tight_layout(); plt.show()

## §8. LLM vs human strata — the core comparison

How models fare on the human-defined hard cases: records with **no listed sense**
(novelty `0`), **low annotator confidence**, **high-confidence annotator
disagreement** (genuine ambiguity), and records that needed **adjudication**.
These are the strata the analysis plan (§7) requires reported separately.

In [ ]:
# Strata summary: pooled accuracy + best-provider accuracy per stratum.
df["min_conf"] = pd.to_numeric(df["min_conf"], errors="coerce")
df["mean_conf"] = pd.to_numeric(df["mean_conf"], errors="coerce")
masks = {
    "ALL (analyzable)":            df.index == df.index,
    "novelty (gold=0)":            df.gold_sense_id == "0",
    "consensus":                   df.resolution == "consensus",
    "adjudicated":                 df.resolution == "adjudicated",
    "genuine-ambiguity (hi-conf disagree)": df.genuine_ambiguity == True,
    "low conf (min≤2)":            df.min_conf <= 2,
    "high conf (min≥4)":           df.min_conf >= 4,
    "restored text":               df.restored == True,
    "context truncated":           df.context_truncated != "none",
}
best_mask = (df.provider_label == best_prov) & (df.config == best_cfg)
rows = []
for label, m in masks.items():
    g = df[m]; gb = df[m & best_mask]
    rows.append({"stratum": label, "n_records": g.key.nunique(),
                 "pooled_acc%": round(100*g.correct.mean(), 1),
                 "best_acc%": round(100*gb.correct.mean(), 1) if len(gb) else float("nan"),
                 "vs_ALL_pp": ""})
strata = pd.DataFrame(rows)
allacc = strata.loc[0, "pooled_acc%"]
strata["vs_ALL_pp"] = (strata["pooled_acc%"] - allacc).round(1)
print(f"LLM accuracy by human stratum (best config = {best_prov}/{best_cfg})\n")
print(strata.to_string(index=False))

In [ ]:
# §8a Novelty / 0-assignment: under-abstention. Models rarely answer 0 even where
# gold is 0 (closed-list WSD bias; WSD_trends.md §8).
gold0 = df[df.gold_sense_id == "0"]
pred0_rate_overall = (df.predicted == 0).mean()
print("Novelty / 0-assignment behaviour")
print(f"  gold 0-rate (records):              {gold0.key.nunique()}/{df.key.nunique()} = {gold0.key.nunique()/df.key.nunique():.3%}")
print(f"  LLM 0-prediction rate (all cells):  {pred0_rate_overall:.3%}")
print(f"  accuracy on the {gold0.key.nunique()} novelty records (LLM answered 0): pooled {100*gold0.correct.mean():.1f}%")
print(f"  per-provider 0-prediction rate / novelty-recall:")
for prov, g in df.groupby("provider_label"):
    g0 = g[g.gold_sense_id == "0"]
    print(f"    {prov:24s} predicts 0 on {100*(g.predicted==0).mean():5.2f}% of all cells; "
          f"recovers {100*g0.correct.mean():5.1f}% of novelty")
print("\nThe gap (gold 0-rate >> LLM 0-rate) is the documented under-abstention finding, reported as-is.")

In [ ]:
# §8b accuracy by annotator confidence; §8c genuine-ambiguity; §8d adjudicated — with significance.
print("Accuracy by min annotator confidence (pooled):")
cb = df.groupby("min_conf").correct.agg(["mean", "size"])
cb["records"] = df.groupby("min_conf").key.nunique()
cb["acc%"] = (100*cb["mean"]).round(1)
print(cb[["acc%", "records"]].to_string())

# paired-ish significance for the binary strata (unpaired here → bootstrap diff, plus a note)
def acc_ci(mask):
    g = df[mask]; n, k = len(g), int(g.correct.sum())
    lo, hi = bootstrap_ci_binary(k, n, n_boot=BOOTSTRAP_N, rng=np.random.default_rng(RNG_SEED))
    return 100*k/n, 100*lo, 100*hi
for label, m in [("genuine-ambiguity", df.genuine_ambiguity == True),
                 ("adjudicated", df.resolution == "adjudicated"),
                 ("consensus", df.resolution == "consensus")]:
    a, lo, hi = acc_ci(m)
    print(f"\n  {label:18s}: acc {a:.1f}% [{lo:.1f}, {hi:.1f}]  (n_records={df[m].key.nunique()})")
print("\nExpected ordering: consensus > adjudicated > genuine-ambiguity. A model gap on the")
print("genuine-ambiguity stratum is defensible — even confident human experts disagreed there.")

### §8e. LLM ↔ human alignment — where they agree and diverge

Using one clean LLM answer per record (the best config), three questions:
(1) how LLM correctness breaks down by *human*-agreement category; (2) on
adjudicated records, whether the LLM sided with the curator's **gold** or with
the **overruled** annotator; and (3) the two asymmetric cases the brief calls
out — **LLM right where annotators split**, and **LLM wrong where annotators
were unanimous and confident**.

In [ ]:
bsub = df[(df.provider_label == best_prov) & (df.config == best_cfg)].set_index("key").copy()

def _hcat(r):
    if r.gold_sense_id == "0":            return "novelty (0)"
    if r.genuine_ambiguity == True:       return "genuine-ambiguity (hi-conf split)"
    if r.resolution == "adjudicated":     return "adjudicated (other)"
    return "consensus"
bsub["hcat"] = bsub.apply(_hcat, axis=1)
ct = bsub.groupby("hcat").agg(n=("correct", "size"), acc=("correct", "mean"))
ct["llm_acc%"] = (100 * ct["acc"]).round(1)
print(f"LLM (best config {best_prov}/{best_cfg}) correctness by human-agreement category\n")
print(ct[["n", "llm_acc%"]].to_string())

# (2) adjudicated siding: LLM pred vs {gold(curator) / overruled annotator / neither}
adj = bsub[(bsub.resolution == "adjudicated") & bsub.ann_senses.str.contains(r"\|")
           & bsub.predicted_sid.notna()].copy()
def _side(r):
    parts = {p for p in r.ann_senses.split("|") if p}
    if r.predicted_sid == r.gold_sense_id:  return "matches gold (curator's pick)"
    if r.predicted_sid in parts:            return "sides with OVERRULED annotator"
    return "neither (a third sense)"
adj["side"] = adj.apply(_side, axis=1)
print(f"\nOn {len(adj)} adjudicated records (two distinct annotator senses), the best LLM:")
for k, v in adj["side"].value_counts().items():
    print(f"  {v:4d} ({v/len(adj):4.0%})  {k}")

# (3) the asymmetric cells
split = bsub[bsub.hcat.str.startswith("genuine")]
clear_miss = bsub[(bsub.hcat == "consensus") & (bsub.min_conf >= 4) & (~bsub.correct)]
print(f"\n>>> LLM OK, annotators NOT: best LLM matches curated gold on {100*split.correct.mean():.0f}% "
      f"of the {len(split)} records where two CONFIDENT annotators chose different senses")
print(f"    (the model 'resolves' a split humans couldn't — its answer agrees with the adjudicated gold).")
print(f">>> annotators OK, LLM NOT: {len(clear_miss)} 'clean misses' — annotators UNANIMOUS and min_conf≥4,")
print(f"    yet the best LLM is wrong. The least excusable errors; stepped through in §10c.")
CLEAR_MISS_KEYS = clear_miss.index.tolist()

## §9. Sentence length & context covariates

Accuracy as a function of sentence length (word count), context truncation, and
text restoration. Length is the headline covariate the brief calls out; the
context-truncation split keeps any future +prev gain from being confounded by
paragraph-edge records.

In [ ]:
core = df.copy()   # df is already the CORE headline slice
core["n_words"] = pd.to_numeric(core["n_words"], errors="coerce")
# quintile bins of sentence length
core["len_bin"] = pd.qcut(core["n_words"], 5, duplicates="drop")
lb = core.groupby("len_bin", observed=True).correct.agg(["mean", "size"])
lb["records"] = core.groupby("len_bin", observed=True).key.nunique()
lb["acc%"] = (100*lb["mean"]).round(1)
print("Accuracy by sentence-length quintile (CORE)\n")
print(lb[["acc%", "records"]].to_string())
rho = core[["n_words", "correct"]].corr(method="spearman").iloc[0, 1]
print(f"\nSpearman ρ(n_words, correct) = {rho:.3f}  (per-cell)")

print("\nAccuracy by context_truncated stratum:")
ct = core.groupby("context_truncated").correct.mean().mul(100).round(1)
ctn = core.groupby("context_truncated").key.nunique()
for k in ["none", "left", "right", "both"]:
    if k in ct.index: print(f"  {k:6s}: {ct[k]:.1f}%  (records {ctn[k]})")
print(f"\nrestored vs original text: restored {100*core[core.restored].correct.mean():.1f}% "
      f"vs original {100*core[~core.restored].correct.mean():.1f}%")
print("  ('restored' = sentence-boundary/segmentation corrections — NOT OCR/content repair. The")
print("   gap is a SOURCE confound: restored records are 35% of hartnoch, the easiest source, and")
print("   concentrate in high-accuracy imperator/dux. Not evidence that correction 'helps'.)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([str(i) for i in lb.index], lb["acc%"], color="#55a868")
ax.set_xlabel("sentence-length quintile (words)"); ax.set_ylabel("accuracy %")
ax.set_title("Accuracy vs sentence length"); plt.xticks(rotation=20, ha="right", fontsize=8)
fig.tight_layout(); plt.show()

## §10. Error analysis — which records are hard, and why

A per-record **difficulty** score = fraction of provider×config cells that got
it wrong. Then: what covariates predict difficulty, the consensus-failure
records (every cell wrong), and the most frequent sense confusions.

In [ ]:
# Per-record difficulty matrix (loader helper) + source-metadata covariates.
PER_REC = E.per_record_matrix(df)
PER_REC = PER_REC.join(_meta.set_index("siglum")[["genre", "author"]], on="siglum")
ncell = int(df.groupby("key").size().iloc[0])
print(f"records: {len(PER_REC)} | mean difficulty {PER_REC.difficulty.mean():.3f} | "
      f"consensus failures (all {ncell} cells wrong): {(PER_REC.difficulty==1).sum()} | "
      f"all-correct: {(PER_REC.difficulty==0).sum()}")
def _d(m): return PER_REC[m].difficulty.mean()
print("\nDifficulty by covariate (mean difficulty; higher = harder):")
print(f"  novelty (gold=0):     {_d(PER_REC.gold_sense_id=='0'):.3f}  vs rest {_d(PER_REC.gold_sense_id!='0'):.3f}")
print(f"  genuine-ambiguity:    {_d(PER_REC.genuine_ambiguity==True):.3f}  vs rest {_d(PER_REC.genuine_ambiguity!=True):.3f}")
print(f"  adjudicated:          {_d(PER_REC.resolution=='adjudicated'):.3f}  vs consensus {_d(PER_REC.resolution=='consensus'):.3f}")
print(f"  low conf (min≤2):     {_d(PER_REC.min_conf<=2):.3f}  vs high (≥4) {_d(PER_REC.min_conf>=4):.3f}")
print(f"  restored:             {_d(PER_REC.restored==True):.3f}  vs original {_d(PER_REC.restored!=True):.3f}")
print(f"  context truncated:    {_d(PER_REC.context_truncated!='none'):.3f}  vs full {_d(PER_REC.context_truncated=='none'):.3f}")
print(f"  genre=polemics:       {_d(PER_REC.genre=='polemics'):.3f}  vs treatise {_d(PER_REC.genre=='treatise'):.3f}")
print(f"  Spearman ρ(n_words, difficulty) = {PER_REC[['n_words','difficulty']].corr(method='spearman').iloc[0,1]:.3f}")

In [ ]:
# Hardest records: every cell wrong. Show the modal (most common) wrong prediction.
hard = PER_REC[PER_REC.difficulty == 1].copy()
def modal_pred(key):
    g = df[df.key == key]
    pm = g.predicted_sid.dropna()
    return pm.mode().iloc[0] if len(pm) else "—"
hard["modal_pred_sid"] = [modal_pred(k) for k in hard.index]
hard = hard.sort_values(["lemma"])
print(f"Consensus-failure records (all {df.groupby('key').size().iloc[0]} cells wrong): {len(hard)}\n")
with pd.option_context("display.max_colwidth", 110, "display.width", 220):
    print(hard.reset_index()[["lemma", "gold_sense_id", "modal_pred_sid", "min_conf",
                              "genuine_ambiguity", "sentence"]]
          .assign(sentence=lambda d: d.sentence.str.slice(0, 100)).head(25).to_string(index=False))

In [ ]:
# Top sense-confusion pairs (gold_sid → predicted_sid on wrong cells), per lemma.
wrong = df[(df.outcome == "wrong") & df.predicted_sid.notna() & (df.gold_sense_id != df.predicted_sid)]
conf = (wrong.groupby(["lemma", "gold_sense_id", "predicted_sid"]).size()
             .rename("n").reset_index().sort_values("n", ascending=False))
print("Top 25 sense-confusion pairs (pooled over all cells)\n")
print(conf.head(25).to_string(index=False))

### §10b. Covariate effect charts (incl. source metadata)

In [ ]:
# Effect-size ranking: Δ accuracy (pp) for each covariate stratum vs overall.
overall = df.correct.mean()
nw = pd.to_numeric(df.n_words, errors="coerce")
strata = {
    "novelty (0)":           df.gold_sense_id == "0",
    "genuine-ambiguity":     df.genuine_ambiguity == True,
    "adjudicated":           df.resolution == "adjudicated",
    "consensus":             df.resolution == "consensus",
    "low conf (min≤2)":      df.min_conf <= 2,
    "high conf (min≥4)":     df.min_conf >= 4,
    "restored text":         df.restored == True,
    "context truncated":     df.context_truncated != "none",
    "genre=polemics":        df.genre == "polemics",
    "genre=treatise":        df.genre == "treatise",
    "long sent (top 20%)":   nw >= nw.quantile(0.8),
}
eff = pd.Series({k: 100 * (df[m].correct.mean() - overall) for k, m in strata.items()}).sort_values()
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.barh(eff.index, eff.values, color=["#c44e52" if v < 0 else "#55a868" for v in eff.values])
ax.axvline(0, c="k", lw=0.8); ax.set_xlabel("Δ accuracy vs overall (pp)")
ax.set_title(f"Covariate effects on LLM accuracy (overall {100*overall:.1f}%)")
for i, v in enumerate(eff.values):
    ax.text(v + (0.4 if v >= 0 else -0.4), i, f"{v:+.1f}", va="center",
            ha="left" if v >= 0 else "right", fontsize=7)
fig.tight_layout(); plt.show()

In [ ]:
# Provider × covariate accuracy heatmap.
covs = {"all": df.index == df.index, "novelty0": df.gold_sense_id == "0",
        "gen-ambig": df.genuine_ambiguity == True, "adjud": df.resolution == "adjudicated",
        "lowconf": df.min_conf <= 2, "hiconf": df.min_conf >= 4,
        "restored": df.restored == True, "ctx-trunc": df.context_truncated != "none",
        "polemics": df.genre == "polemics", "treatise": df.genre == "treatise"}
H = pd.DataFrame({c: 100 * df[m].groupby("provider_label").correct.mean().reindex(PROVIDER_ORDER)
                  for c, m in covs.items()})
fig, ax = plt.subplots(figsize=(11, 3.6))
im = ax.imshow(H.values, cmap="RdYlGn", vmin=20, vmax=100, aspect="auto")
ax.set_xticks(range(len(H.columns))); ax.set_xticklabels(H.columns, rotation=40, ha="right", fontsize=8)
ax.set_yticks(range(len(H.index))); ax.set_yticklabels(H.index, fontsize=8)
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        v = H.values[i, j]
        if v == v: ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7)
ax.set_title("Provider × covariate accuracy (%)"); fig.colorbar(im, ax=ax); fig.tight_layout(); plt.show()

In [ ]:
# Per-source accuracy (4 sources) — CONFOUNDED with lemma, descriptive only.
src = df.groupby("siglum").agg(acc=("correct", "mean"), n_records=("key", "nunique"))
src = src.join(_meta.set_index("siglum")[["author", "genre", "time"]])
src["acc%"] = (100 * src.acc).round(1)
print("Per-source accuracy (only 4 sources; confounded with which lemmas each supplies)\n")
print(src[["author", "genre", "time", "n_records", "acc%"]].sort_values("acc%").to_string())
print("\nPeriod is ~uniform (all 1665–1678), so it is not analysed as a covariate. Source/genre")
print("differences mostly track lemma composition (e.g. comitia/episcopus are near single-source).")

### §10c. Interactive drill-down & cohorts

Filter the per-record matrix to a cohort and walk through it as text
(`records_where` → `inspect_records`), or use the dropdown picker (live in
Jupyter/VSCode; a static top-N table when run headless). This is the tool for
reading *why* a record is hard — the brief's "picker to choose challenging
quotations".

In [ ]:
# Cohort A — novelty records no provider/config solved (n_correct == 0):
print("COHORT: novelty (gold=0) records that every cell got wrong\n")
E.inspect_records(E.records_where(PER_REC, gold_sense_id="0", n_correct_eq=0), df, max_n=3)

In [ ]:
# Cohort B — the "clean misses" from §8e: confident human consensus, LLM hardest.
clean = PER_REC[(PER_REC.resolution == "consensus") & (PER_REC.min_conf >= 4)].sort_values("difficulty", ascending=False)
print(f"COHORT: confident-consensus records, hardest first ({len(clean)} total; showing 3)\n")
E.inspect_records(clean, df, max_n=3)

In [ ]:
# Interactive picker — hardest records first. Live dropdown in Jupyter/VSCode;
# a static top-N table when executed headless (nbconvert).
E.record_picker(PER_REC, df, top_n=60, sort_by="difficulty")

## §11. Experiment 1 ↔ Experiment 2 comparison

Same task framing, **shifted domain and gold provenance**: Exp 1 scored against
SLS dictionary citations (gold = the sense the citation illustrates); Exp 2
scores against human annotation of *running corpus* lines. The comparison
anchors on the 5 providers shared with Exp 1 Phase B (`eval-QBB`, filtered to
l1 / meta-off / plain — the matrix that matches CORE). It is **unpaired**
(different items), so reported as accuracy deltas with CIs, descriptively.

In [ ]:
# Aggregate provider-matched comparison.
try:
    _, df1 = E.load_eval(eval_id("phase_b_plain"))
    df1["correct"] = df1["outcome"] == "correct"
    df1 = df1[(df1.granularity == "l1") & (df1.show_metadata == False)]
    shared = [p for p in PROVIDER_ORDER if p in set(df1.provider_label)]
    cmp_rows = []
    rng = np.random.default_rng(RNG_SEED)
    for prov in shared:
        g2 = df[df.provider_label == prov]; g1 = df1[df1.provider_label == prov]
        a2, _, _ = acc_ci(df.provider_label == prov)
        n1, k1 = len(g1), int(g1.correct.sum())
        lo1, hi1 = bootstrap_ci_binary(k1, n1, n_boot=BOOTSTRAP_N, rng=rng)
        cmp_rows.append({"provider": prov,
                         "exp1_PhaseB_acc%": round(100*k1/n1, 1),
                         "exp2_CORE_acc%": round(a2, 1),
                         "Δ(exp2−exp1)pp": round(a2 - 100*k1/n1, 1),
                         "exp1_MFS": 0.487, "exp2_MFS_micro": round(mfs_micro, 3)})
    cmp = pd.DataFrame(cmp_rows)
    print("Aggregate provider-matched comparison (l1 / meta-off / plain, 4 templates × 2 def)\n")
    print(cmp.to_string(index=False))
    print(f"\nCAVEAT: unpaired (different items, different lemma sets, dictionary vs corpus gold).")
    print(f"Exp 2 sits on a much higher MFS floor ({mfs_micro:.2f} vs Exp 1 0.49), so a higher raw")
    print(f"accuracy does NOT mean the task is easier in the disambiguation sense — read vs MFS.")
    EXP1_OK = True
except Exception as e:
    print(f"Exp 1 Phase B eval not available ({e}). Skipping cross-experiment comparison.")
    EXP1_OK = False

In [ ]:
# Matched-lemma panel: the 4 lemmas evaluated in BOTH experiments.
if EXP1_OK:
    shared_lemmas = sorted(set(df.lemma_uc) & set(df1.lemma.str.upper()))
    print(f"Lemmas evaluated in both experiments: {shared_lemmas}\n")
    rows = []
    for L in shared_lemmas:
        g2 = df[df.lemma_uc == L]; g1 = df1[df1.lemma.str.upper() == L]
        rows.append({"lemma": L, "exp1_acc%": round(100*g1.correct.mean(), 1),
                     "exp2_acc%": round(100*g2.correct.mean(), 1),
                     "exp1_n": g1.shape[0], "exp2_n": g2.shape[0],
                     "Δpp": round(100*(g2.correct.mean()-g1.correct.mean()), 1)})
    print(pd.DataFrame(rows).to_string(index=False))
    print("\nMatched on lemma + provider + template + def_language + l1 inventory; only the")
    print("gold provenance and text type differ. 4 lemmas → directional, not inferential.")
else:
    print("(skipped — Exp 1 data unavailable)")

## §12. Paper-ready outputs

In [ ]:
# Headline table → LaTeX (best config per provider).
paper = best.reset_index()[["provider", "config", "acc%", "ci_lo%", "ci_hi%", "kappa", "dMFS_pp"]]
latex = paper.to_latex(index=False, na_rep="—", float_format="%.1f",
    caption=("Experiment 2 (CORE) per-provider best-config accuracy with 95\\% bootstrap CIs, "
             "sense-id Cohen's $\\kappa$ vs human gold, and gain over the micro most-frequent-sense "
             f"baseline ({100*mfs_micro:.1f}\\%). Corpus-line sense assignment, 837 records, 14 lemmas."),
    label="tab:exp2-llm-headline")
OUTPUT_DIR = Path(os.environ.get('DIALEXPOL_OUTPUT_DIR', Path.cwd() / 'outputs'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'exp2_llm_headline.tex').write_text(latex, encoding='utf-8')
display(paper)

print("HEADLINE NUMBERS (CORE)\n")
print(f"  best config: {best_prov} / {best_cfg} — {best.loc[best_prov,'acc%']}% "
      f"[{best.loc[best_prov,'ci_lo%']}, {best.loc[best_prov,'ci_hi%']}], κ={best.loc[best_prov,'kappa']}")
print(f"  micro-MFS floor {100*mfs_micro:.1f}%, macro-MFS {100*mfs_macro:.1f}% — best model Δ {best.loc[best_prov,'dMFS_pp']:+.1f}pp (micro)")
print(f"  best-config κ vs gold {best.loc[best_prov,'kappa']} vs annotator pooled κ 0.84 (both lemma-blocked, comparable)")
print(f"  novelty under-abstention: gold 0-rate {gold0.key.nunique()/df.key.nunique():.1%} vs LLM 0-rate {pred0_rate_overall:.1%}")
g_amb = df[df.genuine_ambiguity==True]; g_adj = df[df.resolution=="adjudicated"]
print(f"  genuine-ambiguity acc {100*g_amb.correct.mean():.1f}% vs adjudicated {100*g_adj.correct.mean():.1f}% vs consensus {100*df[df.resolution=='consensus'].correct.mean():.1f}%")
print("\nwrote LaTeX → analysis/exp2_llm_headline.tex")
print("Methodology notes: accuracy is read against a high MFS floor; κ/confusion use reconstructed")
print("sense ids (per-record number shuffle); EXTENDED axes (context/metadata) pending the full run.")

## §13. How to read this notebook (statistical orientation)

- **Accuracy** is the exact-match rate vs the curated human gold, pooled over the
  CORE cells unless stated. Always read it against the **MFS floor** (macro/micro
  ≈ 0.82): on this MFS-dominated corpus a high accuracy is not by itself evidence
  of disambiguation skill — the signal is in the low-MFS and hard strata.
- **Bootstrap 95% CIs** (10K resamples) quantify sampling noise on a rate. Cells
  whose CIs overlap (§2 "tied for best") are not distinguishable at this N.
- **McNemar's test** is the *paired* test for two conditions on the same records
  (templates, def_language, context): it uses only the discordant pairs
  (b = first-right/second-wrong, c = first-wrong/second-right).
- **Holm–Bonferroni** controls the family-wise error rate across a family of
  tests (e.g. the 6 template pairs) — read `p_holm`, not raw `p`, when scanning a
  family.
- **κ uses the reconstructed sense ids**, not the per-record-shuffled numbers, and
  is lemma-blocked — directly comparable to the annotator κ in `exp2_iaa.ipynb`.
- **Difficulty** (§10) = fraction of the provider×config cells that got a record
  wrong; it is a consensus signal across models, not a per-model metric.
- The **exp1↔exp2** comparison (§11) and the **source-metadata** breakdowns (§10b)
  are *descriptive*: the former is unpaired (different items/gold), the latter is
  confounded with lemma (only 4 sources). Neither is significance-tested.

---

### Cross-references
- [`exp2_iaa.ipynb`](exp2_iaa.ipynb) — the gold standard and annotation reliability this scores against.
- [`exp2_runs.md`](exp2_runs.md) — run manifest (CORE complete; EXTENDED in progress).
- [`exp2_analysis_plan.md`](exp2_analysis_plan.md) §§7–9 — the analysis design this implements.
- [`the `run` column in llm_predictions/*.csv`](the `run` column in llm_predictions/*.csv) — eval ids (`exp2_core`, `exp2_extended`, `phase_b_plain`).